# The implementation of the 3rd training pipeline, the "Back and Forth"
In this pipeline after initial normal preprocessing steps such as:

    1. Dropping duplicate rows
    2. Applying action mapping
    3. Applying port bucketing
    4. Applying log transformation to all numeric features
We then, before training, excluded features such as "Bytes" and "Packets" as these are a result of a summation between their respective *sent* and *received* components.

After training this CTGAN model the numerical features were inverse transformed (after initial scaling) and all derivable features were recalculated in both the real and artificial datasets.

In [12]:
from pathlib import Path
import pandas as pd
import numpy as np
import logging
from sdv.metadata import SingleTableMetadata
from sdv.single_table import CTGANSynthesizer

ModuleNotFoundError: No module named 'sdv.tabular'

In [4]:
logging.basicConfig(level=logging.INFO)

In [5]:
root = Path.cwd().parent 
data_path = root / "Real Data" / "log2.csv"
df = pd.read_csv(data_path)

In [6]:
df = df.drop_duplicates()

action_mapping = {
    'allow': 0,
    'deny': 1,
    'drop': 2,
    'reset-both': 3
}
df['Action'] = df['Action'].map(action_mapping)

def bucket_port(port):
    if pd.isna(port):
        return 'unknown'
    if 0 <= port <= 1023:
        return 'well_known'
    elif 1024 <= port <= 49151:
        return 'registered'
    elif 49152 <= port <= 65535:
        return 'ephemeral'
    return 'other'

for col in ['Source Port', 'Destination Port', 'NAT Source Port', 'NAT Destination Port']:
    df[f'{col} Category'] = df[col].apply(bucket_port)

In [8]:
cols_to_scale = [
    'Bytes Sent',
    'Bytes Received',
    'Elapsed Time (sec)',
    'pkts_sent',
    'pkts_received'
]

df[cols_to_scale] = df[cols_to_scale].apply(lambda x: np.log1p(x))

In [17]:
df_for_comparison = df

In [9]:
columns_to_exclude = ['Packets', 'Bytes'] # exclude derivable features, which are recalculated after training
df_for_ctgan = df.drop(columns=columns_to_exclude, errors='ignore')

In [10]:
metadata = SingleTableMetadata()
metadata.detect_from_dataframe(df_for_ctgan)

INFO:sdv.metadata.single_table:Detected metadata:
INFO:sdv.metadata.single_table:{
    "METADATA_SPEC_VERSION": "SINGLE_TABLE_V1",
    "columns": {
        "Source Port": {
            "sdtype": "numerical"
        },
        "Destination Port": {
            "sdtype": "numerical"
        },
        "NAT Source Port": {
            "sdtype": "numerical"
        },
        "NAT Destination Port": {
            "sdtype": "numerical"
        },
        "Action": {
            "sdtype": "categorical"
        },
        "Bytes Sent": {
            "sdtype": "numerical"
        },
        "Bytes Received": {
            "sdtype": "numerical"
        },
        "Elapsed Time (sec)": {
            "sdtype": "numerical"
        },
        "pkts_sent": {
            "sdtype": "numerical"
        },
        "pkts_received": {
            "sdtype": "numerical"
        },
        "Source Port Category": {
            "sdtype": "categorical"
        },
        "Destination Port Category": {
        

In [8]:
synthesizer = CTGANSynthesizer(
    metadata=metadata,
    epochs=50,
    verbose=True
)

synthesizer.fit(df_for_ctgan)
synthesizer.save('ctgan_model_backAndForth.pkl')

C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:144: FutureWarning: The 'SingleTableMetadata' is deprecated. Please use the new 'Metadata' class for synthesizers.
  warnings.warn(DEPRECATION_MSG, FutureWarning)
C:\Users\Карен Оганян\AppData\Local\Programs\Python\Python311\Lib\site-packages\sdv\single_table\base.py:122: UserWarning: We strongly recommend saving the metadata using 'save_to_json' for replicability in future SDV versions.
  warnings.warn(
INFO:SingleTableSynthesizer:{'EVENT': 'Instance', 'TIMESTAMP': datetime.datetime(2025, 5, 31, 15, 35, 50, 762788), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_b034e472288c4326834f525fe3f04606'}
INFO:SingleTableSynthesizer:{'EVENT': 'Fit', 'TIMESTAMP': datetime.datetime(2025, 5, 31, 15, 35, 50, 762788), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_b034e472288c4326834f525fe3f04606', 'TOTAL NUMBER 

In [13]:
# loaded_model = CTGANSynthesizer.load('ctgan_model_backAndForth.pkl')

INFO:SingleTableSynthesizer:{'EVENT': 'Load', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 19, 58, 41, 218106), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_b034e472288c4326834f525fe3f04606'}


In [14]:
synthetic_data = loaded_model.sample(len(df))

INFO:SingleTableSynthesizer:{'EVENT': 'Sample', 'TIMESTAMP': datetime.datetime(2025, 6, 1, 19, 58, 47, 648333), 'SYNTHESIZER CLASS NAME': 'CTGANSynthesizer', 'SYNTHESIZER ID': 'CTGANSynthesizer_1.20.1_b034e472288c4326834f525fe3f04606', 'TOTAL NUMBER OF TABLES': 1, 'TOTAL NUMBER OF ROWS': 57170, 'TOTAL NUMBER OF COLUMNS': 14}


In [15]:
cols_to_scale = [
    'Bytes Sent',
    'Bytes Received',
    'pkts_sent',
    'pkts_received'
]

synthetic_data[cols_to_scale] = np.expm1(synthetic_data[cols_to_scale])

synthetic_data[cols_to_scale] = synthetic_data[cols_to_scale].round().astype(int)

synthetic_data['Bytes'] = np.log1p(synthetic_data['Bytes Sent'] + synthetic_data['Bytes Received'])
synthetic_data['Packets'] = np.log1p(synthetic_data['pkts_sent'] + synthetic_data['pkts_received'])
synthetic_data['Bytes_log'] = np.log1p(synthetic_data['Bytes'])
synthetic_data['pkts_log'] = np.log1p(synthetic_data['pkts_sent'] + synthetic_data['pkts_received'])

eps = 1e-6

synthetic_data['pkts_ratio'] = np.log1p(synthetic_data['pkts_sent'] / (synthetic_data['pkts_received'] + eps))
synthetic_data['Bytes Ratio'] = np.log1p(synthetic_data['Bytes Sent'] / (synthetic_data['Bytes Received'] + eps))

In [16]:
synthetic_data.to_csv('BackAndForth.csv')

In [20]:
# Compute derived features in real dataset 
df_for_comparison['Bytes'] = df_for_comparison['Bytes Sent'] + df_for_comparison['Bytes Received']
df_for_comparison['Packets'] = df_for_comparison['pkts_sent'] + df_for_comparison['pkts_received']
df_for_comparison['Bytes_log'] = np.log1p(df_for_comparison['Bytes'])
df_for_comparison['pkts_log'] = np.log1p(df_for_comparison['Packets'])

eps = 1e-6

df_for_comparison['pkts_ratio'] = np.log1p(df_for_comparison['pkts_sent'] / (df_for_comparison['pkts_received'] + eps))
df_for_comparison['Bytes Ratio'] = np.log1p(df_for_comparison['Bytes Sent'] / (df_for_comparison['Bytes Received'] + eps))

In [63]:
df_for_comparison.to_csv('BackAndForthREAL.csv')

In [21]:
df_for_comparison.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes,Bytes Sent,Bytes Received,Packets,Elapsed Time (sec),pkts_sent,pkts_received,Source Port Category,Destination Port Category,NAT Source Port Category,NAT Destination Port Category,Bytes_log,pkts_log,pkts_ratio,Bytes Ratio
0,57222,53,54587,53,0,3.406586,1.714496,1.692090,1.053178,1.489299,0.526589,0.526589,ephemeral,well_known,ephemeral,well_known,1.483100,0.719389,0.693146,0.699746
1,56258,3389,56258,3389,0,4.329653,2.125655,2.203998,2.417862,1.358505,1.223156,1.194706,ephemeral,registered,ephemeral,registered,1.673286,1.229015,0.704983,0.675214
2,6881,50321,43265,50321,0,3.511384,1.754252,1.757132,1.053178,2.090638,0.526589,0.526589,registered,ephemeral,registered,ephemeral,1.506604,0.719389,0.693146,0.692327
3,50553,3389,50553,3389,0,4.258109,2.112840,2.145268,2.287031,1.358505,1.162283,1.124748,ephemeral,registered,ephemeral,registered,1.659771,1.189985,0.709695,0.685560
4,50002,443,45848,443,0,4.666893,2.284583,2.382310,2.664031,1.343703,1.291725,1.372307,ephemeral,well_known,registered,well_known,1.734641,1.298564,0.663347,0.672423


In [22]:
synthetic_data.head()

,Source Port,Destination Port,NAT Source Port,NAT Destination Port,Action,Bytes Sent,Bytes Received,Elapsed Time (sec),pkts_sent,pkts_received,Source Port Category,Destination Port Category,NAT Source Port Category,NAT Destination Port Category,Bytes,Packets,Bytes_log,pkts_log,pkts_ratio,Bytes Ratio
0,38388,220,44233,165,0,102,94,30,1,1,ephemeral,well_known,registered,well_known,196,2,5.283204,1.098612,0.693147,0.734820
1,65040,158,42,0,2,69,0,0,1,0,ephemeral,well_known,well_known,well_known,69,1,4.248495,0.693147,13.815512,18.049617
2,61906,22438,179,0,1,64,0,0,1,0,ephemeral,registered,well_known,well_known,64,1,4.174387,0.693147,13.815512,17.974394
3,51704,58,55908,333,0,6655,716,69,24,7,ephemeral,well_known,ephemeral,well_known,7371,31,8.905444,3.465736,1.488077,2.331628
4,10679,308,5557,229,0,105,150,30,18,1,registered,well_known,registered,well_known,255,19,5.545177,2.995732,2.944438,0.530628
